# 11 Download BRO groundwater levels

<!-- navigation -->
[<kbd>← 10 Import time series from Excel</kbd>](10%20Import%20time%20series%20from%20Excel.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>12 Download river levels (Rijkswaterstaat) →</kbd>](12%20Download%20river%20levels%20%28Rijkswaterstaat%29.ipynb)

**Optional**: instead of, or in addition to, notebook 10

Downloads the groundwater levels of the BRO monitoring wells around each transect into its pastastore, and drafts a piezometer table.

| | |
|---|---|
| **Input** | rectangle per transect in the transects metadata (`xmin_extent` .. `ymax_extent`) |
| **Output** | pastastore per transect; draft piezometer table `piezometers_from_bro.xlsx` |

Downloads the groundwater level series of all BRO monitoring wells in the rectangle of each transect, with hydropandas, and stores them in the pastastore of the transect. Times are converted to Dutch winter time (UTC+1). The draft piezometer table lists the downloaded filters; copy the rows you want to use to `piezometers.xlsx`, fill in `modellayer` and `x_crosssection`, and run notebook 01 again.

In [ ]:
from pathlib import Path

import timflow_levee_toolkit as tlt

import hydropandas as hpd
import pandas as pd
print(f"timflow_levee_toolkit {tlt.__version__}")

In [ ]:
## UPDATE FOR YOUR PROJECT: the project folder (with project_settings.toml and the data)
project_dir = Path.cwd().parent / "examples" / "papendrecht"

project = tlt.load_project(project_dir)
units = project.units
project

In [ ]:
## UPDATE FOR YOUR PROJECT: transects and download period
metadata = project.metadata()
transects = project.transects()  # or a selection, e.g. ["Papendrecht West"]

# Download period: None = all available data. The model period is chosen later, in notebook 51,
# from the figures of the complete series. The example downloads only one week, to keep it small.
tmin = pd.Timestamp("2025-12-01")  # None: from the first measurement
tmax = pd.Timestamp("2025-12-08")  # None: up to the last measurement
print(f"Transects: {transects}, download period: {tmin} - {tmax}")

## 1. Download and store in the pastastore
_BRO identifiers of series can change. Use `overwrite = True` to replace series that are already in the pastastore._

In [ ]:
overwrite = False  # True: replace series that are already in the pastastore

drafts = []
meta_by_transect = metadata.set_index("transect")
for transect in transects:
    m = meta_by_transect.loc[transect]
    extent = [m.get(c) for c in ("xmin_extent", "xmax_extent", "ymin_extent", "ymax_extent")]
    if any(pd.isna(v) for v in extent):
        print(f"{transect}: no rectangle (xmin_extent .. ymax_extent) in the metadata, skipped")
        continue

    oc = hpd.read_bro(extent=extent, tmin=tmin, tmax=tmax, keep_all_obs=False)
    print(f"{transect}: {len(oc)} BRO series with measurements")
    if len(oc) == 0:
        continue

    series = pd.concat({name: o.iloc[:, 0] for name, o in oc.obs.items()}, axis=1)
    series_metadata = pd.DataFrame(oc.drop(columns="obs"))
    pstore = project.open_pastastore(transect)
    added = tlt.import_timeseries(pstore, series, metadata=series_metadata, overwrite=overwrite)
    print(f"{transect}: {len(added)} series added, {len(pstore.oseries_names)} in the pastastore")
    drafts.append(tlt.piezometer_table_from_obs(oc, transect))

    # Section of the wells: filter depths and measured heads
    folder = project.transect_dir(transect, "timeseries")
    folder.mkdir(parents=True, exist_ok=True)
    try:
        oc.name = transect
        oc.plots.section_plot(fn_save=folder / "bro_wells_section.png")
    except Exception as e:
        print(f"{transect}: no section plot ({type(e).__name__}: {e})")

## 2. Draft piezometer table

In [ ]:
draft = pd.concat(drafts, ignore_index=True) if drafts else pd.DataFrame()
if not draft.empty:
    fn_draft = project.general_dir / "piezometers_from_bro.xlsx"
    tlt.export_to_excel(draft, fn_draft, index=False)
    print(f"Saved {project.relpath(fn_draft)}: copy the rows you need to piezometers.xlsx, "
          "fill in modellayer and x_crosssection, and run notebook 01 again")
draft

---

<!-- navigation -->
[<kbd>← 10 Import time series from Excel</kbd>](10%20Import%20time%20series%20from%20Excel.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>12 Download river levels (Rijkswaterstaat) →</kbd>](12%20Download%20river%20levels%20%28Rijkswaterstaat%29.ipynb)